# Feature Track 3: Maintain a Conversation

In a multi-turn dialogue the assistant must remember what was said earlier. This notebook explores three mechanisms for passing conversation history to a RAG agent:

| # | Strategy | Idea |
|---|---|---|
| 1 | **Full history** | Append every exchange verbatim to the context window |
| 2 | **History summarization** | Compress prior turns into a short summary before each query |
| 3 | **Query reformulation** | Rewrite the user query to be self-contained before retrieval |

---

## Setup

**Prerequisites:** `conversational_toolkit` and `backend` must be installed in editable mode.
For **OpenAI** set `OPENAI_API_KEY`. For **Ollama** start `ollama serve` and pull the model.

In [1]:
from loguru import logger
import sys

from conversational_toolkit.llms.base import LLMMessage, Roles, MessageContent
from conversational_toolkit.agents.base import QueryWithContext
from conversational_toolkit.embeddings.openai import OpenAIEmbeddings
from conversational_toolkit.vectorstores.chromadb import ChromaDBVectorStore

from sme_kt_zh_collaboration_rag.feature0_baseline_rag import (
    load_chunks,
    build_vector_store,
    build_llm,
    build_agent,
    SYSTEM_PROMPT,
    EMBEDDING_MODEL,
    VS_PATH,
)

logger.remove()
logger.add(sys.stderr, level="WARNING")  # hides DEBUG

# Choose your LLM backend (only needed for the final answer cells)
BACKEND = "openai"  # "ollama", "openai", or "qwen"
FORCE_REBUILD = False  # set True to re-embed from scratch and rebuild the vector store

embedding_model = OpenAIEmbeddings(model_name=EMBEDDING_MODEL)
print(f"Embedding model: {EMBEDDING_MODEL}")

if FORCE_REBUILD or not VS_PATH.exists():
    chunks = load_chunks(max_files=5)
    chunks = [c for c in chunks if c.mime_type.startswith("text")]
    vector_store = await build_vector_store(
        chunks, embedding_model, db_path=VS_PATH, reset=FORCE_REBUILD
    )
else:
    # Vector store already exists -> open it without re-embedding
    vector_store = ChromaDBVectorStore(db_path=str(VS_PATH))
    print(
        f"Reusing existing vector store at {VS_PATH} ({vector_store.collection.count()} chunks)"
    )

llm = build_llm(backend=BACKEND)
print(f"LLM backend: {BACKEND}")

Embedding model: text-embedding-3-small
Reusing existing vector store at /Users/pkoerner/Desktop/Kanton_Zurich/sme-kt-zh-collaboration-rag/backend/db/data_vs.db (1663 chunks)
LLM backend: openai


In [2]:
agent = build_agent(
    vector_store,
    embedding_model,
    llm,
    top_k=5,
    system_prompt=SYSTEM_PROMPT,
)

### Single-turn baseline

A single query with an empty history -> the starting point before any multi-turn logic is introduced.

In [3]:
query = "Which pallets in our portfolio have a third-party verified EPD?"
query_with_context = QueryWithContext(query=query, history=[])

response = await agent.answer(query_with_context)

print("Answer:")
print(f"{response.content}")
print(f"Sources ({len(response.sources)}):")
for src in response.sources:
    source_file = src.metadata.get("source_file", "?")
    print(f"  {source_file!r}  |  {src.title!r}")

Answer:
[MessageContent(type='text', text='The pallets in your portfolio that have a third-party verified EPD are:\n\n1. **IPG** - Products 50-100 and 50-101: Compliant, EPDs on file.\n2. **CPR System** - Product 32-100: Compliant, EPD on file.\n3. **Relicyc** - Product 32-103: Compliant, EPD on file.\n4. **StabilPlastik** - Product 32-105: Compliant, EPD on file.\n5. **Redbox** - Product 11-100: Compliant, EPD on file.\n6. **Grupak** - Product 11-101: Compliant, EPD on file.\n\nThese products have valid third-party verified EPDs as per the compliance status reported (source: document excerpt 425ec050-51a6-4c57-9800-b0b6e90ef240).', image_url=None)]
Sources (5):
  'EPD_tape_IPG_wateractivated.pdf'  |  '## EPD Programme Information'
  'EPD_tape_IPG_hotmelt.pdf'  |  '## EPD Programme Information'
  'EPD_cardboard_redbox_cartonpallet.pdf'  |  '## Third-party verification'
  'EPD_cardboard_grupak_corrugated.pdf'  |  '## Third-party verification'
  'ART_internal_procurement_policy.pdf'  |  

---

## Strategy 1: Full History

The simplest approach is to pass the entire conversation history to the agent on every turn. Each exchange (user query + assistant answer) is appended as a pair of `LLMMessage` objects and sent verbatim in the next request.

**Trade-off:** Works well for short conversations. As the history grows it consumes more of the context window, increases cost, and can degrade retrieval quality because the retriever must rank chunks against an increasingly long prompt.

> Note: retrieved source chunks are intentionally *not* included in the history. Storing full chunks would make the context balloon quickly.

Three queries are run in sequence. After each answer the user and assistant turns are appended to `history`, which is passed to the agent on the next call. The third query ("Make minutes of our conversation") only makes sense with history — it would fail without it.

In [ ]:
query_l = [
    "Which pallets in our portfolio have a third-party verified EPD?",
    "Rewrite what you just said as a pirate and very concisely.",
    "Make minutes of our conversation so far, you are 'pAIrate' and I'm AIvan",
]

history: list[LLMMessage] = []

for query in query_l:
    response = await agent.answer(QueryWithContext(query=query, history=history))

    print("_" * 80)
    print("\nHistory:")
    for msg in history:
        print(f"{msg.role.value}: {msg.content[0].text}")
        print("-" * 40)
    print("\nAnswer:")
    print(f"{response.content[0].text}")
    print(f"\nSources ({len(response.sources)}):")
    for src in response.sources:
        source_file = src.metadata.get("source_file", "?")
        print(f"  {source_file!r} | {src.title!r}")

    response_content = response.content[0].text
    history.append(
        LLMMessage(content=[MessageContent(type="text", text=query)], role=Roles.USER)
    )
    history.append(
        LLMMessage(
            content=[MessageContent(type="text", text=response_content)],
            role=Roles.ASSISTANT,
        )
    )

2026-03-25 11:57:05.744 | INFO     | conversational_toolkit.embeddings.openai:get_embeddings:37 - OpenAI embeddings shape: (1, 1024)


________________________________________________________________________________

History:

Answer:
The pallets in your portfolio that have a third-party verified EPD are:

1. **IPG** - Products 50-100 and 50-101: Compliant, EPDs on file.
2. **CPR System** - Product 32-100: Compliant, EPD on file.
3. **Relicyc** - Product 32-103: Compliant, EPD on file.
4. **StabilPlastik** - Product 32-105: Compliant, EPD on file.
5. **Redbox** - Product 11-100: Compliant, EPD on file.
6. **Grupak** - Product 11-101: Compliant, EPD on file.

These products have valid third-party verified EPDs as per the compliance status reported (source: excerpt from the document).

Sources (5):
  'EPD_tape_IPG_wateractivated.pdf' | '## EPD Programme Information'
  'EPD_tape_IPG_hotmelt.pdf' | '## EPD Programme Information'
  'EPD_cardboard_redbox_cartonpallet.pdf' | '## Third-party verification'
  'EPD_cardboard_grupak_corrugated.pdf' | '## Third-party verification'
  'ART_internal_procurement_policy.pdf' | '## 3.2 

2026-03-25 11:57:11.304 | DEBUG    | conversational_toolkit.llms.openai:generate:121 - Completion: ChatCompletion(id='chatcmpl-DNGD8Ev8URTNYviQIp3WAlEsS6hN3', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Rewrite the information about the pallets in your portfolio that have a third-party verified EPD as a pirate and very concisely.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1774436230, model='gpt-4o-mini-2024-07-18', object='chat.completion', service_tier='default', system_fingerprint='fp_ca3e7d71bf', usage=CompletionUsage(completion_tokens=26, prompt_tokens=403, total_tokens=429, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))
2026-03-25 11:57:11.305 | INFO     | conversational_toolkit.llms.o

________________________________________________________________________________

History:
user: Which pallets in our portfolio have a third-party verified EPD?
----------------------------------------
assistant: The pallets in your portfolio that have a third-party verified EPD are:

1. **IPG** - Products 50-100 and 50-101: Compliant, EPDs on file.
2. **CPR System** - Product 32-100: Compliant, EPD on file.
3. **Relicyc** - Product 32-103: Compliant, EPD on file.
4. **StabilPlastik** - Product 32-105: Compliant, EPD on file.
5. **Redbox** - Product 11-100: Compliant, EPD on file.
6. **Grupak** - Product 11-101: Compliant, EPD on file.

These products have valid third-party verified EPDs as per the compliance status reported (source: excerpt from the document).
----------------------------------------

Answer:
Arrr! Here be the pallets in yer treasure trove with third-party verified EPDs:

1. **IPG** - Products 50-100, 50-101: Aye, EPDs on file.
2. **CPR System** - Product 32-100: Aye,

2026-03-25 11:57:17.804 | DEBUG    | conversational_toolkit.llms.openai:generate:121 - Completion: ChatCompletion(id='chatcmpl-DNGDEe7vleJOauGMvCEvKDSg892aF', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="Make minutes of our conversation so far, you are 'pAIrate' and I'm AIvan, including the pallets in our portfolio that have a third-party verified EPD.", refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1774436236, model='gpt-4o-mini-2024-07-18', object='chat.completion', service_tier='default', system_fingerprint='fp_3a2472b832', usage=CompletionUsage(completion_tokens=35, prompt_tokens=628, total_tokens=663, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))
2026-03-25 11:57:17.805 | INFO     | convers

________________________________________________________________________________

History:
user: Which pallets in our portfolio have a third-party verified EPD?
----------------------------------------
assistant: The pallets in your portfolio that have a third-party verified EPD are:

1. **IPG** - Products 50-100 and 50-101: Compliant, EPDs on file.
2. **CPR System** - Product 32-100: Compliant, EPD on file.
3. **Relicyc** - Product 32-103: Compliant, EPD on file.
4. **StabilPlastik** - Product 32-105: Compliant, EPD on file.
5. **Redbox** - Product 11-100: Compliant, EPD on file.
6. **Grupak** - Product 11-101: Compliant, EPD on file.

These products have valid third-party verified EPDs as per the compliance status reported (source: excerpt from the document).
----------------------------------------
user: Rewrite what you just said as a pirate and very concisely.
----------------------------------------
assistant: Arrr! Here be the pallets in yer treasure trove with third-party verif

---

## Strategy 2: History Summarization

Instead of sending the full history, a cheap LLM (`gpt-4o-mini`) is called before each turn to compress prior exchanges into a single summary message. Only that summary — not the raw history — is injected into the next request as a `SYSTEM` message.

**Trade-off:** Keeps the context window small and cost low, but lossy — fine-grained details from earlier turns may be dropped. Good when the conversation is long but only the gist of prior turns matters.

In [6]:
async def summarize_history(history: list[LLMMessage]) -> list[MessageContent]:
    llm_summarization = build_llm("openai", model_name="gpt-4o-mini")

    summarization_system_prompt = "You are a helpful assistant that summarizes conversations between a user and an assistant. Do not answer any question, just summarize the conversation in a concise way. The user is called 'User' and the assistant is called 'Assistant'."
    messages: list[LLMMessage] = [
        LLMMessage(
            content=[MessageContent(type="text", text=summarization_system_prompt)],
            role=Roles.SYSTEM,
        ),
        *history,
    ]

    msg = await llm_summarization.generate(conversation=messages)
    return msg.content

In [7]:
query_l = [
    "Which pallets in our portfolio have a third-party verified EPD?",
    "Rewrite what you just said as a pirate and very concisely.",
    "What is the current customer request?",
]

history: list[LLMMessage] = []
summaries: list[list[MessageContent]] = []

for query in query_l:
    if len(history):
        summary_msg = await summarize_history(history)
        print(f"Summary: {summary_msg[0].text}")
        summaries.append(summary_msg)
        print("x" * 80)
        history_tmp = [LLMMessage(content=summary_msg, role=Roles.SYSTEM)]
    else:
        history_tmp = []

    response = await agent.answer(QueryWithContext(query=query, history=history_tmp))
    response_content = response.content[0].text

    print("Answer:")
    print(f"{response_content}")
    print(f"\nSources ({len(response.sources)}):")
    for src in response.sources:
        source_file = src.metadata.get("source_file", "?")
        print(f"  {source_file!r}  |  {src.title!r}")

    history.append(
        LLMMessage(content=[MessageContent(type="text", text=query)], role=Roles.USER)
    )
    history.append(
        LLMMessage(
            content=[MessageContent(type="text", text=response_content)],
            role=Roles.ASSISTANT,
        )
    )

    print("-" * 80)

Answer:
The pallets in your portfolio that have a third-party verified EPD are:

1. **IPG** - Products 50-100 and 50-101 (Compliant, EPDs on file).
2. **CPR System** - Product 32-100 (Compliant, EPD on file).
3. **Relicyc** - Product 32-103 (Compliant, EPD on file).
4. **StabilPlastik** - Product 32-105 (Compliant, EPD on file).
5. **Redbox** - Product 11-100 (Compliant, EPD on file).
6. **Grupak** - Product 11-101 (Compliant, EPD on file).

These products have valid third-party verified EPDs as per the compliance status reported (source: document excerpt 425ec050-51a6-4c57-9800-b0b6e90ef240 and document excerpt bfbd5f6e-0bab-41c3-acdd-82178f00a0e5).

Sources (5):
  'EPD_tape_IPG_wateractivated.pdf'  |  '## EPD Programme Information'
  'EPD_tape_IPG_hotmelt.pdf'  |  '## EPD Programme Information'
  'EPD_cardboard_redbox_cartonpallet.pdf'  |  '## Third-party verification'
  'EPD_cardboard_grupak_corrugated.pdf'  |  '## Third-party verification'
  'ART_internal_procurement_policy.pdf'  |

### Summarization loop

On the first turn no summary exists yet, so the history is empty. From the second turn onward `summarize_history` compresses all prior exchanges and the result is passed as the sole context message.

In [8]:
history

[LLMMessage(content=[MessageContent(type='text', text='Which pallets in our portfolio have a third-party verified EPD?', image_url=None)], role=<Roles.USER: 'user'>, tool_calls=None, tool_call_id=None, name=None),
 LLMMessage(content=[MessageContent(type='text', text='The pallets in your portfolio that have a third-party verified EPD are:\n\n1. **IPG** - Products 50-100 and 50-101 (Compliant, EPDs on file).\n2. **CPR System** - Product 32-100 (Compliant, EPD on file).\n3. **Relicyc** - Product 32-103 (Compliant, EPD on file).\n4. **StabilPlastik** - Product 32-105 (Compliant, EPD on file).\n5. **Redbox** - Product 11-100 (Compliant, EPD on file).\n6. **Grupak** - Product 11-101 (Compliant, EPD on file).\n\nThese products have valid third-party verified EPDs as per the compliance status reported (source: document excerpt 425ec050-51a6-4c57-9800-b0b6e90ef240 and document excerpt bfbd5f6e-0bab-41c3-acdd-82178f00a0e5).', image_url=None)], role=<Roles.ASSISTANT: 'assistant'>, tool_calls=Non

In [9]:
summaries

[[MessageContent(type='text', text='User inquired about which pallets in their portfolio have a third-party verified Environmental Product Declaration (EPD). The assistant provided a list of specific products from various brands that meet this criterion, confirming their compliance and the availability of EPDs on file.', image_url=None)],
 [MessageContent(type='text', text='The user inquired about which pallets in their portfolio have a third-party verified Environmental Product Declaration (EPD). The assistant provided a list of specific pallets that meet this criterion, including details about their compliance status and EPD availability.', image_url=None)]]

---

## Strategy 3: Query Reformulation

When a user says "What is the meaning of code?" in the context of an EPD conversation, the retriever has no way to know they mean "EPD code" without the history. Query reformulation solves this by asking an LLM to rewrite the query into a self-contained form before embedding it for retrieval.

This is already wired into the agent via `make_query_standalone` in `conversational_toolkit.utils.retriever`. Enabling `DEBUG` logging below makes the reformulation step visible.

In [10]:
logger.add(sys.stderr, level="DEBUG")

2

In [11]:
query_l = [
    "Hi, I'm Ivan! What is the pallet which has a 3rd-party verified EPD?",
    "What is the meaning of code?",
]

history: list[LLMMessage] = []

for query in query_l:
    response = await agent.answer(QueryWithContext(query=query, history=history))
    response_content = response.content[0].text

    print("Answer:")
    print(f"{response_content}")
    print(f"\nSources ({len(response.sources)}):")
    for src in response.sources:
        source_file = src.metadata.get("source_file", "?")
        print(f"  {source_file!r}  |  {src.title!r}")

    history.append(
        LLMMessage(content=[MessageContent(type="text", text=query)], role=Roles.USER)
    )
    history.append(
        LLMMessage(
            content=[MessageContent(type="text", text=response_content)],
            role=Roles.ASSISTANT,
        )
    )

    print("-" * 80)

2026-03-25 11:40:25.765 | INFO     | conversational_toolkit.embeddings.openai:get_embeddings:37 - OpenAI embeddings shape: (1, 1024)


Answer:
Based on the provided excerpts, there are multiple pallets with third-party verified EPDs. Here are the details:

1. **Pallet with EPD verified by WAP Sustainability Consulting**:
   - Third-party verifier: Maggie Wildnauer, Brad McAllister.
   - Verification is in accordance with ISO 14025:2006.
   - The EPD is valid until 2023-11-08 (source: EPD Programme Information, excerpt id 'e8844211-022a-4f59-b304-9b1f9972a38f').

2. **Pallet with EPD verified by Bureau Veritas Italia S.p.A.**:
   - This verification is also according to ISO 14025:2006.
   - The certification body is accredited by Accredia (source: Third-party verification, excerpt id 'f9dbc2af-fc87-46fc-83f7-5b9caf64a5c8').

3. **Pallet with EPD verified by IK Ingeniería SL**:
   - Third-party verifier: Ruben Carnerero.
   - This verification is conducted without a pre-verified LCA/EPD tool (source: Third-party verification, excerpt id '01c48b79-a86f-44d0-8995-46df63c2df5d').

4. **Pallet with EPD verified by Silcert, 

2026-03-25 11:40:35.245 | DEBUG    | conversational_toolkit.llms.openai:generate:121 - Completion: ChatCompletion(id='chatcmpl-DNFx4fAJhYFi2699LVmGkI3MCRL8T', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='What is the meaning of a pallet with a third-party verified EPD?', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1774435234, model='gpt-4o-mini-2024-07-18', object='chat.completion', service_tier='default', system_fingerprint='fp_ca3e7d71bf', usage=CompletionUsage(completion_tokens=15, prompt_tokens=645, total_tokens=660, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))
2026-03-25 11:40:35.246 | INFO     | conversational_toolkit.llms.openai:generate:122 - LLM Usage: CompletionUsage(completion_token

Answer:
A pallet with a third-party verified Environmental Product Declaration (EPD) means that the environmental impact data and claims associated with that pallet have been independently assessed and verified by an accredited certification body. This verification process is conducted according to the standards set out in ISO 14025:2006, which governs the development of EPDs.

The significance of this verification includes:

1. **Credibility**: The data presented in the EPD is considered reliable because it has been evaluated by an independent third party, reducing the risk of bias or inaccuracies that could arise from self-declared claims.

2. **Transparency**: The EPD provides detailed information about the environmental impacts of the pallet, allowing consumers and businesses to make informed decisions based on standardized data.

3. **Comparability**: EPDs that are verified by third parties can be compared against other EPDs, provided they adhere to the same Product Category Rules

---------------------------